In [ ]:
# CELL 1: Setup
import sys
sys.path.append('..')

import os
import torch
import numpy as np
import random
import matplotlib.pyplot as plt

from configs.config import Config
from data.splits import get_datasets
from models.prototypical_segmentation import PrototypicalSegmentation
from training.prototypical_trainer import PrototypicalTrainer
from configs.results_utils import save_kshot_results, print_kshot_results
from configs.model_utils import load_model_weights

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

Config.create_dirs()
print(f"✓ Device: {Config.DEVICE}")

ModuleNotFoundError: No module named 'torch'

In [ ]:
# CELL 2: Data Loading
train_dataset, val_dataset, test_dataset = get_datasets(Config)

print(f"✓ Train samples: {len(train_dataset)}")
print(f"✓ Val samples:   {len(val_dataset)}")
print(f"✓ Test samples:  {len(test_dataset)}")

In [ ]:
# CELL 3: Create Model + Load Pretrained Encoder
model = PrototypicalSegmentation(
    encoder_name=Config.ENCODER_NAME,
    in_channels=Config.IN_CHANNELS,
    num_classes=Config.NUM_CLASSES,
).to(Config.DEVICE)

load_model_weights(model.unet, Config.CHECKPOINT_DIR, 'best_model.pth', Config.DEVICE)
print(f"✓ Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# CELL 4: Sanity Check
trainer = PrototypicalTrainer(
    model=model,
    config=Config,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
)

# Test one episode
episode = trainer.sample_episode(train_dataset, k_shot=5, n_query=10)
print(f"✓ Support: {episode['support_images'].shape}")
print(f"✓ Query:   {episode['query_images'].shape}")

# Test prototype extraction
support_imgs = episode['support_images'].to(Config.DEVICE)
support_masks = episode['support_masks'].to(Config.DEVICE)
prototypes = model.compute_prototypes(support_imgs, support_masks)
print(f"✓ Extracted {len(prototypes)} prototypes, shape: {prototypes[0].shape}")

In [ ]:
# CELL 5: Train Prototypical Networks
print("=" * 60)
print("STARTING PROTOTYPICAL NETWORKS TRAINING")
print("=" * 60)

history = trainer.train(
    num_episodes=1000,
    k_shot=5,
    n_query=Config.N_QUERY,
)

# Training Curve
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(history['train_loss'], label='Train Loss')
ax.plot(history['val_loss'], label='Val Loss')
ax.set_xlabel('Episode (×100)')
ax.set_ylabel('Dice Loss')
ax.set_title('Prototypical Networks Training')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(Config.RESULTS_DIR, 'prototypical_training_curve.png'), dpi=150)
plt.show()

In [ ]:
# CELL 6: Evaluate at Different k Values
k_shot_results = trainer.evaluate_k_shot(
    k_values=Config.K_SHOT_VALUES,
    num_episodes=50,
)

print_kshot_results(k_shot_results, Config.K_SHOT_VALUES, "PROTOTYPICAL NETWORKS K-SHOT RESULTS")
save_kshot_results(k_shot_results, Config.RESULTS_DIR, 'prototypical_kshot_results.json')

In [ ]:
# CELL 7: Learning Curve
k_values = Config.K_SHOT_VALUES
means = [k_shot_results[k]['mean'] for k in k_values]
stds = [k_shot_results[k]['std'] for k in k_values]

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(k_values, means, yerr=stds, marker='o', capsize=5,
            linewidth=2, markersize=8, color='#4CAF50')
ax.set_xlabel('k (number of support examples)', fontsize=12)
ax.set_ylabel('Mean Tumor Dice Score', fontsize=12)
ax.set_title('Prototypical Networks: Dice vs. k', fontsize=14)
ax.set_xticks(k_values)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

for k, dice in zip(k_values, means):
    ax.text(k, dice + 0.02, f'{dice:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(Config.RESULTS_DIR, 'prototypical_learning_curve.png'), dpi=150)
plt.show()